In [3]:
import os
import shutil
import time
from datetime import datetime, timedelta

# مسیر مبدا و مقصد
source_dir = r"D:\PT\edge_rl_3"
dest_dir = r"C:\Users\Admin\Desktop\edge_rl_3"

# محاسبه زمان ۳ ساعت قبل
cutoff_time = datetime.now() - timedelta(hours=3)

# تبدیل به timestamp برای مقایسه با mtime
cutoff_timestamp = cutoff_time.timestamp()

# پیمایش دایرکتوری مبدا
for root, dirs, files in os.walk(source_dir):
    for file in files:
        if file.endswith(".py"):
            file_path = os.path.join(root, file)
            
            # دریافت زمان آخرین ویرایش
            mtime = os.path.getmtime(file_path)
            
            # اگر فایل در ۳ ساعت اخیر ویرایش شده
            if mtime >= cutoff_timestamp:
                # محاسبه مسیر نسبی نسبت به مبدا
                rel_path = os.path.relpath(root, source_dir)
                dest_subdir = os.path.join(dest_dir, rel_path)
                
                # ساخت پوشه مقصد در صورت نیاز
                os.makedirs(dest_subdir, exist_ok=True)
                
                # مسیر کامل فایل مقصد
                dest_file = os.path.join(dest_subdir, file)
                
                # کپی فایل
                shutil.copy2(file_path, dest_file)
                print(f"کپی شد: {file_path} -> {dest_file}")

کپی شد: D:\PT\edge_rl_3\algorithms\base.py -> C:\Users\Admin\Desktop\edge_rl_3\algorithms\base.py
کپی شد: D:\PT\edge_rl_3\algorithms\greedy\greedy_algorithm.py -> C:\Users\Admin\Desktop\edge_rl_3\algorithms\greedy\greedy_algorithm.py
کپی شد: D:\PT\edge_rl_3\algorithms\hpa\hpa_algorithm.py -> C:\Users\Admin\Desktop\edge_rl_3\algorithms\hpa\hpa_algorithm.py
کپی شد: D:\PT\edge_rl_3\algorithms\ppo\env.py -> C:\Users\Admin\Desktop\edge_rl_3\algorithms\ppo\env.py
کپی شد: D:\PT\edge_rl_3\algorithms\ppo\ppo_algorithm.py -> C:\Users\Admin\Desktop\edge_rl_3\algorithms\ppo\ppo_algorithm.py
کپی شد: D:\PT\edge_rl_3\algorithms\voila\voila_algorithm.py -> C:\Users\Admin\Desktop\edge_rl_3\algorithms\voila\voila_algorithm.py
کپی شد: D:\PT\edge_rl_3\common\metrics.py -> C:\Users\Admin\Desktop\edge_rl_3\common\metrics.py
کپی شد: D:\PT\edge_rl_3\common\models.py -> C:\Users\Admin\Desktop\edge_rl_3\common\models.py
کپی شد: D:\PT\edge_rl_3\k8s_adapter\realtime_dispatcher.py -> C:\Users\Admin\Desktop\edge_rl

In [ ]:
# check_placement.py
from common.config import CFG
from common.models import Server
from data.loader import load_train
from algorithms.ppo.optimal_placement import aggregate_training_demand, solve_optimal_server_selection

servers = {}
for sid, info in CFG.server_info.items():
    prof = CFG.server_profiles[info["profile"]]
    servers[sid] = Server(id=sid, profile=info["profile"], lat=info["lat"], long=info["long"],
                           capacity=info["capacity"], p_idle=prof["p_idle"], p_max=prof["p_max"])

train_events = load_train()
demand_points = aggregate_training_demand(train_events)

# می‌تونی وزن‌ها رو عوض کنی تا ببینی نتیجه چطور تغییر می‌کنه
selected = solve_optimal_server_selection(
    servers, demand_points,
    w_count=0.2, w_energy=1.0, w_distance=0.2,
)

total_cpu_needed = sum(s["cpu_demand"] for s in CFG.services_info.values())
total_selected_capacity = sum(servers[sid].capacity for sid in selected)
total_idle_power = sum(servers[sid].p_idle for sid in selected)

print(f"تعداد نقاط تقاضا: {len(demand_points)}")
print(f"تعداد سرور انتخاب‌شده: {len(selected)}")
print(f"مجموع ظرفیت: {total_selected_capacity} (نیاز: {total_cpu_needed})")
print(f"مجموع توان idle: {total_idle_power}W")
print(f"سرورهای انتخاب‌شده: {sorted(selected)}")
for sid in sorted(selected):
    info = CFG.server_info[sid]
    print(f"  سرور {sid}: profile={info['profile']}, capacity={info['capacity']}, "
          f"p_idle={CFG.server_profiles[info['profile']]['p_idle']}W")

تعداد نقاط تقاضا: 1487
تعداد سرور انتخاب‌شده: 2
مجموع ظرفیت: 260 (نیاز: 248)
مجموع توان idle: 150W
سرورهای انتخاب‌شده: [2, 9]
  سرور 2: profile=edge_small, capacity=60, p_idle=40W
  سرور 9: profile=large, capacity=200, p_idle=110W
